In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.utils import resample
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Masking, LSTM, Dense, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tqdm import tqdm

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab. Ensure Google Drive is accessible.")

# === Configuration ===
config = {
    "PARTICIPANT_DIR": "/content/drive/MyDrive/audio_files/AA",
    "MAPPING_DIR": "/content/drive/MyDrive/merged participants",
    "MAPPING_FILE": "/content/drive/MyDrive/merged participants/Copy of appetite levels mapping.csv",
    "TEST_SIZE": 0.2,
    "VALIDATION_SPLIT": 0.2,
    "BATCH_SIZE": 16,
    "EPOCHS": 50,
    "PATIENCE": 5,
    "MASK_VALUE": 0.0,
    "LSTM_UNITS": 64,
    "MAX_SEQ_LENGTH": 500,
    "RANDOM_STATE": 42
}

def compute_sample_weights(y):
    """Compute sample weights with stronger emphasis on rare values."""
    bins = np.round(y).astype(int).clip(0, 3)
    class_counts = np.bincount(bins, minlength=4)
    total_samples = len(y)

    weights = np.zeros_like(y, dtype=float)
    for i in range(4):
        if class_counts[i] > 0:
            weights[bins == i] = (total_samples / (class_counts[i] + 1)) ** 1.5
        else:
            weights[bins == i] = 0.0

    weights /= np.mean(weights)  # Normalize to stabilize training
    print("Sample weight distribution:", np.unique(weights, return_counts=True))
    return weights

def oversample_rare_values(X_mfcc, X_egemaps, y, target_count=75):
    """Oversample samples with PHQ8_5_Appetite values 2 and 3."""
    bins = np.round(y).astype(int).clip(0, 3)
    X_mfcc_oversampled, X_egemaps_oversampled, y_oversampled = [], [], []

    for bin_val in range(4):
        mask = bins == bin_val
        X_mfcc_bin = X_mfcc[mask]
        X_egemaps_bin = X_egemaps[mask]
        y_bin = y[mask]

        if len(y_bin) == 0:
            continue

        if bin_val in [2, 3]:  # Oversample rare values
            X_mfcc_bin, X_egemaps_bin, y_bin = resample(
                X_mfcc_bin, X_egemaps_bin, y_bin,
                replace=True,
                n_samples=target_count,
                random_state=config["RANDOM_STATE"]
            )

        X_mfcc_oversampled.append(X_mfcc_bin)
        X_egemaps_oversampled.append(X_egemaps_bin)
        y_oversampled.append(y_bin)

    X_mfcc = np.concatenate(X_mfcc_oversampled, axis=0)
    X_egemaps = np.concatenate(X_egemaps_oversampled, axis=0)
    y = np.concatenate(y_oversampled, axis=0)

    indices = np.arange(len(y))
    np.random.shuffle(indices)
    return X_mfcc[indices], X_egemaps[indices], y[indices]

def load_and_preprocess_data():
    print("Loading data from Google Drive folders...")

    if not os.path.exists(config["PARTICIPANT_DIR"]):
        raise FileNotFoundError(f"Participant folder not found at {config['PARTICIPANT_DIR']}.")
    if not os.path.exists(config["MAPPING_DIR"]):
        raise FileNotFoundError(f"Mapping folder not found at {config['MAPPING_DIR']}.")

    print(f"Files in {config['PARTICIPANT_DIR']}:")
    print(os.listdir(config["PARTICIPANT_DIR"])[:10], "...")
    print(f"Files in {config['MAPPING_DIR']}:")
    print(os.listdir(config["MAPPING_DIR"]))

    mapping_file = config["MAPPING_FILE"]
    if not os.path.exists(mapping_file):
        raise FileNotFoundError(f"Mapping file not found at {mapping_file}")
    print(f"Using mapping file: {mapping_file}")

    try:
        map_df = pd.read_csv(mapping_file)
        print("Mapping file loaded successfully")
        print("First few rows of mapping file:")
        print(map_df.head())
        print("Columns:", list(map_df.columns))
        print("PHQ8_5_Appetite value counts:")
        print(map_df['PHQ8_5_Appetite'].value_counts())

        if 'Participant' not in map_df.columns or 'PHQ8_5_Appetite' not in map_df.columns:
            raise ValueError(f"Required columns 'Participant' and 'PHQ8_5_Appetite' not found.")

        map_df['PHQ8_5_Appetite'] = pd.to_numeric(map_df['PHQ8_5_Appetite'], errors='coerce')
        map_df = map_df.dropna(subset=['PHQ8_5_Appetite'])
        participant_labels = dict(zip(map_df['Participant'].astype(str), map_df['PHQ8_5_Appetite']))
        print(f"Loaded labels for {len(participant_labels)} participants")
        print("Sample participant labels:", list(participant_labels.items())[:5])
    except Exception as e:
        raise ValueError(f"Error loading mapping file: {str(e)}")

    all_files = glob.glob(os.path.join(config["PARTICIPANT_DIR"], "*.csv"))
    print(f"Found {len(all_files)} CSV files in the participant folder")

    mfcc_files = {}
    egemaps_files = {}

    for f in all_files:
        filename = os.path.basename(f)
        try:
            pid = filename.split('_')[0]
            if 'MFCC' in filename.upper():
                mfcc_files[pid] = f
            elif 'EGEMAPS' in filename.upper():
                egemaps_files[pid] = f
        except Exception as e:
            print(f"Skipping file {filename}: Invalid format ({str(e)})")
            continue

    print(f"Found {len(mfcc_files)} MFCC files and {len(egemaps_files)} eGeMAPS files")

    common_pids = set(mfcc_files.keys()) & set(egemaps_files.keys()) & set(participant_labels.keys())
    print(f"Found {len(common_pids)} complete participant records")

    if not common_pids:
        raise ValueError(
            f"No matching participants found.\n"
            f"File IDs: {list(mfcc_files.keys())[:5]}...\n"
            f"Mapping IDs: {list(participant_labels.keys())[:5]}..."
        )

    mfcc_seqs = []
    egemaps_seqs = []
    labels = []

    imputer_mfcc = SimpleImputer(strategy='mean')
    imputer_egemaps = SimpleImputer(strategy='mean')

    for pid in tqdm(common_pids, desc="Processing participants"):
        try:
            df_mfcc = pd.read_csv(mfcc_files[pid], nrows=config["MAX_SEQ_LENGTH"])
            df_mfcc = df_mfcc.select_dtypes(include=[np.number])
            if df_mfcc.empty:
                print(f"Skipping participant {pid} - empty MFCC data")
                continue

            df_egemaps = pd.read_csv(egemaps_files[pid], nrows=config["MAX_SEQ_LENGTH"])
            df_egemaps = df_egemaps.select_dtypes(include=[np.number])
            if df_egemaps.empty:
                print(f"Skipping participant {pid} - empty eGeMAPS data")
                continue

            seq_mfcc = df_mfcc.values[:config["MAX_SEQ_LENGTH"]]
            seq_egemaps = df_egemaps.values[:config["MAX_SEQ_LENGTH"]]

            for seq, name in [(seq_mfcc, "MFCC"), (seq_egemaps, "eGeMAPS")]:
                if len(seq) < config["MAX_SEQ_LENGTH"]:
                    pad_width = ((0, config["MAX_SEQ_LENGTH"] - len(seq)), (0, 0))
                    seq = np.pad(seq, pad_width, mode='constant', constant_values=config["MASK_VALUE"])
                elif len(seq) > config["MAX_SEQ_LENGTH"]:
                    seq = seq[:config["MAX_SEQ_LENGTH"]]

            mfcc_seqs.append(seq_mfcc)
            egemaps_seqs.append(seq_egemaps)
            labels.append(participant_labels[pid])

        except Exception as e:
            print(f"Error processing participant {pid}: {str(e)}")
            continue

    if not mfcc_seqs:
        raise ValueError("No valid data was processed - check your input files")

    X_mfcc = np.array(mfcc_seqs, dtype='float32')
    X_egemaps = np.array(egemaps_seqs, dtype='float32')
    y = np.array(labels, dtype='float32')

    X_mfcc = imputer_mfcc.fit_transform(X_mfcc.reshape(-1, X_mfcc.shape[-1])).reshape(X_mfcc.shape)
    X_egemaps = imputer_egemaps.fit_transform(X_egemaps.reshape(-1, X_egemaps.shape[-1])).reshape(X_egemaps.shape)

    scaler_mfcc = StandardScaler()
    scaler_egemaps = StandardScaler()

    X_mfcc = scaler_mfcc.fit_transform(X_mfcc.reshape(-1, X_mfcc.shape[-1])).reshape(X_mfcc.shape)
    X_egemaps = scaler_egemaps.fit_transform(X_egemaps.reshape(-1, X_egemaps.shape[-1])).reshape(X_egemaps.shape)

    print("\nData preprocessing complete")
    print(f"MFCC data shape: {X_mfcc.shape}")
    print(f"eGeMAPS data shape: {X_egemaps.shape}")
    print(f"Labels shape: {y.shape}")
    print(f"Label distribution (rounded): {np.bincount(np.round(y).astype(int), minlength=4)}")

    return X_mfcc, X_egemaps, y

def augment_data(X):
    """Apply noise injection to time-series data."""
    noise = np.random.normal(0, 0.01, X.shape)
    return X + noise

def build_model(mfcc_shape, egemaps_shape):
    """Simplified LSTM model to reduce overfitting."""
    mfcc_input = Input(shape=mfcc_shape, name='mfcc_input')
    egemaps_input = Input(shape=egemaps_shape, name='egemaps_input')

    x1 = Masking(mask_value=config["MASK_VALUE"])(mfcc_input)
    x1 = LSTM(config["LSTM_UNITS"], activation='tanh')(x1)
    x1 = BatchNormalization()(x1)
    x1 = Dropout(0.3)(x1)

    x2 = Masking(mask_value=config["MASK_VALUE"])(egemaps_input)
    x2 = LSTM(config["LSTM_UNITS"], activation='tanh')(x2)
    x2 = BatchNormalization()(x2)
    x2 = Dropout(0.3)(x2)

    merged = Concatenate()([x1, x2])
    x = Dense(32, activation='relu')(merged)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='linear')(x)

    model = Model(inputs=[mfcc_input, egemaps_input], outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    return model

def evaluate_per_bin(y_true, y_pred):
    """Evaluate MSE and MAE per rounded value bin (0, 1, 2, 3)."""
    bins_true = np.round(y_true).astype(int).clip(0, 3)
    bins_pred = np.round(y_pred).astype(int).clip(0, 3)

    print("\nPer-bin evaluation:")
    for bin_val in range(4):
        mask = bins_true == bin_val
        if np.sum(mask) > 0:
            mse_bin = np.mean((y_true[mask] - y_pred[mask])**2)
            mae_bin = np.mean(np.abs(y_true[mask] - y_pred[mask]))
            print(f"Bin {bin_val} (n={np.sum(mask)}): MSE={mse_bin:.4f}, MAE={mae_bin:.4f}")
        else:
            print(f"Bin {bin_val} (n=0): No samples")

def cross_validate():
    print("Starting appetite prediction pipeline with cross-validation...")

    try:
        X_mfcc, X_egemaps, y = load_and_preprocess_data()
        kf = KFold(n_splits=5, shuffle=True, random_state=config["RANDOM_STATE"])
        mse_scores, mae_scores, r2_scores = [], [], []
        fold = 1

        for train_idx, test_idx in kf.split(X_mfcc):
            print(f"\nProcessing fold {fold}/5")
            X_train_mfcc, X_test_mfcc = X_mfcc[train_idx], X_mfcc[test_idx]
            X_train_egemaps, X_test_egemaps = X_egemaps[train_idx], X_egemaps[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Oversample rare values
            X_train_mfcc, X_train_egemaps, y_train = oversample_rare_values(
                X_train_mfcc, X_train_egemaps, y_train, target_count=75
            )

            # Compute sample weights
            sample_weights = compute_sample_weights(y_train)

            # Augment training data
            X_train_mfcc_aug = augment_data(X_train_mfcc)
            X_train_egemaps_aug = augment_data(X_train_egemaps)

            print(f"Training samples: {len(y_train)}")
            print(f"Test samples: {len(y_test)}")
            print(f"Training label distribution (rounded): {np.bincount(np.round(y_train).astype(int), minlength=4)}")
            print(f"Test label distribution (rounded): {np.bincount(np.round(y_test).astype(int), minlength=4)}")

            model = build_model(X_train_mfcc.shape[1:], X_train_egemaps.shape[1:])
            print("\nModel architecture:")
            model.summary()

            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=config["PATIENCE"],
                restore_best_weights=True
            )
            lr_scheduler = ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-6
            )
            history = model.fit(
                [X_train_mfcc_aug, X_train_egemaps_aug], y_train,
                sample_weight=sample_weights,
                validation_split=config["VALIDATION_SPLIT"],
                epochs=config["EPOCHS"],
                batch_size=config["BATCH_SIZE"],
                callbacks=[early_stop, lr_scheduler],
                verbose=1
            )

            mse, mae = model.evaluate([X_test_mfcc, X_test_egemaps], y_test, verbose=0)
            predictions = model.predict([X_test_mfcc, X_test_egemaps], verbose=0).flatten()
            r2 = r2_score(y_test, predictions)

            mse_scores.append(mse)
            mae_scores.append(mae)
            r2_scores.append(r2)

            print(f"\nFold {fold} results:")
            print(f"Test MSE: {mse:.4f}")
            print(f"Test MAE: {mae:.4f}")
            print(f"Test R²: {r2:.4f}")
            evaluate_per_bin(y_test, predictions)

            # Save predictions for this fold
            results = pd.DataFrame({
                'True': y_test,
                'Predicted': predictions
            })
            results.to_csv(f"/content/drive/MyDrive/test_predictions_regression_fold_{fold}.csv", index=False)
            print(f"Predictions saved to /content/drive/MyDrive/test_predictions_regression_fold_{fold}.csv")

            print("\nSample predictions:")
            for true, pred in zip(y_test[:10], predictions[:10]):
                print(f"True: {true:.2f}, Predicted: {pred:.2f}")

            fold += 1

        print(f"\nCross-validation results: MSE={np.mean(mse_scores):.4f}±{np.std(mse_scores):.4f}, "
              f"MAE={np.mean(mae_scores):.4f}±{np.std(mae_scores):.4f}, "
              f"R²={np.mean(r2_scores):.4f}±{np.std(r2_scores):.4f}")

    except Exception as e:
        print(f"\nError in cross-validation: {str(e)}")
        raise

if _name_ == "_main_":
    cross_validate()
    print("\nPipeline execution completed!")